In [ ]:
import itertools
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import seaborn as sns
from tqdm.auto import tqdm

In [ ]:
window_size = 0.1
stride = 0.05
outdir = "."

In [ ]:
all_epoch_paths = list(Path("epochs").glob("*.fif"))

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epochs", str(path))[0]
    if subject_name == "EC282":
        # missing ecog data
        continue
    epochs[subject_name] = mne.read_epochs(str(path)).pick("ecog").resample(100)

In [ ]:
global_tmin = 0. # min([epoch.times.min() for epoch in epochs.values()])
global_tmax = max([epoch.times.max() for epoch in epochs.values()])
windows_left = np.arange(global_tmin, global_tmax, stride)
windows_right = np.minimum(global_tmax, windows_left + window_size)
windows = list(zip(windows_left, windows_right))

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline

scores = {}
phoneme_pairs = next(iter(epochs.values())).phoneme_pair.unique()

for phoneme_pair, (tmin, tmax), subject_name in tqdm(list(itertools.product(phoneme_pairs, windows, epochs))):
    epochs_i = epochs[subject_name][f"phoneme_pair == '{phoneme_pair}'"].copy().crop(tmin, tmax)
    if len(epochs_i) == 0:
        continue

    # epochs * channels * time
    X = epochs_i.get_data()
    X = X.reshape(X.shape[0], -1)

    y = epochs_i.metadata.word_end.str[0] == epochs_i.metadata.phoneme_pair.str[0]

    cv_inner = StratifiedKFold(3, shuffle=True)
    cv_outer = StratifiedKFold(3, shuffle=True)

    model = make_pipeline(StandardScaler(), PCA(n_components=0.95),
                          LogisticRegressionCV(Cs=10, cv=cv_inner, max_iter=1000))
    scores_i = cross_val_score(model, X, y, cv=cv_outer, scoring="roc_auc")

    scores[subject_name, phoneme_pair, tmin, tmax] = scores_i
    print(subject_name, phoneme_pair, tmin, tmax, scores_i.mean())

In [ ]:
scores_df = pd.concat({key: pd.Series(scores_i).rename("roc_auc") for key, scores_i in scores.items()},
                      names=["subject", "phoneme_pair", "tmin", "tmax", "fold"])

In [ ]:
scores_df.to_csv(Path(outdir) / "scores.csv")